# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display the dataset metadata summary
meta = dataset.metadata
print(f"\033[1mDataset Name:\033[0m {meta.name}\n")
print(f"\033[1mDescription:\033[0m {meta.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs (always referenced by their `@id`).

In [ ]:
# List all record sets in the dataset, with their @id and the fields/columns contained
record_sets = dataset.metadata.record_sets()

print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}, Name: {rs['name']}")
    print("  Fields/Columns:")
    for field in rs['fields']:
        col_id = field['@id'] if '@id' in field else field.get('id', 'N/A')
        col_name = field.get('name', 'N/A')
        datatype = field.get('dataType', 'N/A')
        print(f"    - {col_id} (name='{col_name}', type={datatype})")
    print("")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis using their `@id`.

In [ ]:
# Step 1. Build a list of all record set @ids
rs_ids = [rs['@id'] for rs in dataset.metadata.record_sets()]

# Step 2. Extract records from each record set, store DataFrames in a dict
dataframes = {}
for rs_id in rs_ids:
    
    try:
        print(f"Extracting records for RecordSet {rs_id} ...")
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Shape: {df.shape}. Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Could not load records for RecordSet {rs_id}: {e}")

# Pick the main record set for analysis (first one)
main_record_set_id = rs_ids[0] if rs_ids else None
if main_record_set_id:
    print(f"\nMain record set columns ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())


## 4. Exploratory Data Analysis (EDA)
We now apply common data processing steps: filtering records based on criteria, normalizing numeric fields, or grouping by a key. All fields/columns are referenced by `@id` as in earlier steps.

In [ ]:
from pandas.api.types import is_numeric_dtype

if main_record_set_id:
    rs = [rs for rs in dataset.metadata.record_sets() if rs['@id'] == main_record_set_id][0]
    main_df = dataframes[main_record_set_id]
    # Try to pick a numeric field. We'll pick the first numeric column found.
    numeric_field = None
    for field in rs['fields']:
        col_id = field['@id'] if '@id' in field else field.get('id', 'N/A')
        if col_id in main_df.columns and is_numeric_dtype(main_df[col_id]):
            numeric_field = col_id
            break

    if not numeric_field:
        # Try to convert a likely numeric column based on common field names
        for field in rs['fields']:
            col_id = field['@id'] if '@id' in field else field.get('id', 'N/A')
            # Try to convert column to numeric if possible
            if col_id in main_df.columns:
                main_df[col_id] = pd.to_numeric(main_df[col_id], errors='coerce')
                if is_numeric_dtype(main_df[col_id]):
                    numeric_field = col_id
                    break

    if numeric_field:
        print(f"Using numeric field (by @id): {numeric_field}")
        # Demo: Filter rows greater than threshold (10)
        threshold = 10
        filtered_df = main_df[main_df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} rows")
        display(filtered_df.head())

        # Normalization
        mean = filtered_df[numeric_field].mean()
        std = filtered_df[numeric_field].std()
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by another field
        group_field = None
        for field in rs['fields']:
            col_id = field['@id'] if '@id' in field else field.get('id', 'N/A')
            # Use first non-numeric string field
            if col_id != numeric_field and col_id in main_df.columns and main_df[col_id].dtype == object:
                group_field = col_id
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean of {numeric_field}):")
            display(grouped_df.head())
        else:
            print("Could not find suitable group field.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No main record set available.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using standard plotting libraries. Field references use their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field:
    # Plot histogram of the selected numeric field
    plt.figure(figsize=(8,4))
    main_df[numeric_field].dropna().hist(bins=30)
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.title(f'Distribution of {numeric_field}')
    plt.show()

    # If group_field exists, plot boxplot
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=main_df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated usage of `mlcroissant` to load and analyze the dataset "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells".
- We loaded the dataset metadata and explored record sets.
- Data was extracted by referencing RecordSet and Field `@id`s.
- Typical EDA was performed: filtering, normalization, and grouping on a numeric field.
- Visualizations provided distributions of important variables.

For advanced research, continue refining filters, exploring relationships between additional fields (using their `@id`), and perform domain-specific statistical analysis.